In [0]:
print("TrustGuard notebook is running successfully!")

spark

TrustGuard notebook is running successfully!


In [0]:
raw_df = spark.table("workspace.default.trustguard_retail_transactions")

display(raw_df.limit(10))

print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))

transaction_id,customer_id,full_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_tier,is_active,transaction_date,store_id,product_id,product_category,item,quantity,unit_price,total_amount,payment_method,transaction_type,created_at,source_system
TXN-00000001,CUST-00951,Karan Sinha,karan.sinha340@gmail.com,+91-6478805564,M,05-17-1993,Bangalore,Karnataka,10/20/22,Bronze,Yes,2024-06-26,ONLINE,PRD-2441,Home & Kitchen,Bedsheet,2,1149.09,2298.18,UPI,SALE,2024-06-26T20:17:00.000Z,MOBILE_APP
TXN-00000002,CUST-00072,Sahil Patel,sahil.patel859@rediffmail.com,7178884276,MALE,11-25-1993,Indore,Madhya Pradesh,2023-09-24,Platinum,TRUE,04/28/24,STORE-044,PRD-3426,Sports,Yoga Mat,5,721.0,3605.0,Net Banking,SALE,2024-04-28T09:02:00.000Z,MOBILE_APP
TXN-00000003,CUST-01036,Harsh Sharma,harsh.sharma622@outlook.com,79596-10945,M,03-12-1992,Hyderabad,Telangana,13-07-2023,Platinum,Y,06/24/24,STORE-010,PRD-6531,Electronics,Power Bank,4,1503.12,6012.48,wallet,REFUND,2024-06-24T22:01:00.000Z,CRM_EXPORT
TXN-00000004,CUST-01382,Sakshi Sharma,sakshi.sharma56@yahoo.com,76711-28657,FEMALE,02-25-2000,Jaipur,Rajasthan,2023-04-07,Silver,1,2024-04-23,ONLINE,PRD-6028,Home & Kitchen,Pressure Cooker,4,1992.0,7968.0,wallet,SALE,2024-04-23T19:46:00.000Z,WEBSITE
TXN-00000005,CUST-02194,Priya Joshi,priya.joshi793@yahoo.com,99930-16643,female,06/04/1985,Bangalore,Karnataka,01-09-2021,Silver,Y,02/12/24,ONLINE,PRD-7016,Stationery,Gel Pen Set,2,139.41,278.82,COD,SALE,2024-02-12T14:39:00.000Z,CRM_EXPORT
TXN-00000006,CUST-00563,Isha Iyer,NULL,6779355497,FEMALE,10/01/1997,Bangalore,Karnataka,12-04-2021,Platinum,1,09-06-2024,ONLINE,PRD-9849,Groceries,Wheat Flour 10kg,5,486.53,2432.65,Credit Card,SALE,2024-06-09T13:09:00.000Z,MOBILE_APP
TXN-00000007,CUST-02027,Karan Joshi,karan.joshi135@rediffmail.com,+91-6109084830,Male,1994-07-04,Bhopal,Madhya Pradesh,21-03-2021,Silver,Yes,2024-01-14,ONLINE,PRD-9228,Beauty,Body Lotion,1,255.62,255.62,NetBanking,SALE,2024-01-14T15:02:00.000Z,MOBILE_APP
TXN-00000008,CUST-00559,Varun Das,varun.das931@rediffmail.com,7455984805,M,11-15-2005,Kolkata,West Bengal,10/12/21,Silver,True,05/06/24,STORE-055,PRD-6037,Fashion,Sneakers,1,2725.05,2725.05,Upi,SALE,2024-05-06T10:41:00.000Z,WEBSITE
TXN-00000009,CUST-00044,Kavya Verma,kavya.verma22@rediffmail.com,95318-84396,F,07-12-1978,Jaipur,Rajasthan,02-07-2023,Bronze,Y,14-02-2024,ONLINE,PRD-6531,NULL,Power Bank,4,1472.47,5889.88,wallet,EXCHANGE,2024-02-14T20:56:00.000Z,MOBILE_APP
TXN-00000010,CUST-01634,Kavya Gupta,kavya.gupta609@gmail.com,+91-9977283871,female,15/04/2005,Kolkata,West Bengal,2021-09-12,null,1,06/26/24,ONLINE,PRD-4677,Electronics,Smart Watch,5,3713.75,18568.75,COD,SALE,2024-06-26T17:32:00.000Z,POS


Rows: 10250
Columns: 24


# TrustGuard: Data Quality Pipeline for Retail Data

This project is built using Python, PySpark, SQL, Delta Lake, and Databricks.

Architecture:

Raw Layer → Clean Layer → Final Layer

The pipeline ingests dirty retail data, validates it, cleans it, separates rejected records, creates final analysis-ready tables, and generates data quality reports.

In [0]:
# Cell 2: Project Setup

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import uuid
from datetime import datetime

# Create unique pipeline run details
run_id = str(uuid.uuid4())
run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", run_id)
print("Run Timestamp:", run_timestamp)

# Use workspace catalog
spark.sql("USE CATALOG workspace")

# Create project schema/database
spark.sql("CREATE SCHEMA IF NOT EXISTS trustguard_db")

# Use project schema/database
spark.sql("USE SCHEMA trustguard_db")

print("Catalog selected: workspace")
print("Schema selected: trustguard_db")

Run ID: 9c3a1eb1-248a-4654-a0d6-b7331ee07f19
Run Timestamp: 2026-07-11 15:22:55
Catalog selected: workspace
Schema selected: trustguard_db


In [0]:
spark.sql("SHOW TABLES IN default").show(truncate=False)

+--------+------------------------------+-----------+
|database|tableName                     |isTemporary|
+--------+------------------------------+-----------+
|default |trustguard_retail_transactions|false      |
+--------+------------------------------+-----------+



In [0]:
# Cell 4: Load uploaded raw table from Databricks Catalog

raw_df = spark.table("workspace.default.trustguard_retail_transactions")

print("Raw row count:", raw_df.count())
print("Raw column count:", len(raw_df.columns))

display(raw_df.limit(10))

Raw row count: 10250
Raw column count: 24


transaction_id,customer_id,full_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_tier,is_active,transaction_date,store_id,product_id,product_category,item,quantity,unit_price,total_amount,payment_method,transaction_type,created_at,source_system
TXN-00000001,CUST-00951,Karan Sinha,karan.sinha340@gmail.com,+91-6478805564,M,05-17-1993,Bangalore,Karnataka,10/20/22,Bronze,Yes,2024-06-26,ONLINE,PRD-2441,Home & Kitchen,Bedsheet,2,1149.09,2298.18,UPI,SALE,2024-06-26T20:17:00.000Z,MOBILE_APP
TXN-00000002,CUST-00072,Sahil Patel,sahil.patel859@rediffmail.com,7178884276,MALE,11-25-1993,Indore,Madhya Pradesh,2023-09-24,Platinum,TRUE,04/28/24,STORE-044,PRD-3426,Sports,Yoga Mat,5,721.0,3605.0,Net Banking,SALE,2024-04-28T09:02:00.000Z,MOBILE_APP
TXN-00000003,CUST-01036,Harsh Sharma,harsh.sharma622@outlook.com,79596-10945,M,03-12-1992,Hyderabad,Telangana,13-07-2023,Platinum,Y,06/24/24,STORE-010,PRD-6531,Electronics,Power Bank,4,1503.12,6012.48,wallet,REFUND,2024-06-24T22:01:00.000Z,CRM_EXPORT
TXN-00000004,CUST-01382,Sakshi Sharma,sakshi.sharma56@yahoo.com,76711-28657,FEMALE,02-25-2000,Jaipur,Rajasthan,2023-04-07,Silver,1,2024-04-23,ONLINE,PRD-6028,Home & Kitchen,Pressure Cooker,4,1992.0,7968.0,wallet,SALE,2024-04-23T19:46:00.000Z,WEBSITE
TXN-00000005,CUST-02194,Priya Joshi,priya.joshi793@yahoo.com,99930-16643,female,06/04/1985,Bangalore,Karnataka,01-09-2021,Silver,Y,02/12/24,ONLINE,PRD-7016,Stationery,Gel Pen Set,2,139.41,278.82,COD,SALE,2024-02-12T14:39:00.000Z,CRM_EXPORT
TXN-00000006,CUST-00563,Isha Iyer,NULL,6779355497,FEMALE,10/01/1997,Bangalore,Karnataka,12-04-2021,Platinum,1,09-06-2024,ONLINE,PRD-9849,Groceries,Wheat Flour 10kg,5,486.53,2432.65,Credit Card,SALE,2024-06-09T13:09:00.000Z,MOBILE_APP
TXN-00000007,CUST-02027,Karan Joshi,karan.joshi135@rediffmail.com,+91-6109084830,Male,1994-07-04,Bhopal,Madhya Pradesh,21-03-2021,Silver,Yes,2024-01-14,ONLINE,PRD-9228,Beauty,Body Lotion,1,255.62,255.62,NetBanking,SALE,2024-01-14T15:02:00.000Z,MOBILE_APP
TXN-00000008,CUST-00559,Varun Das,varun.das931@rediffmail.com,7455984805,M,11-15-2005,Kolkata,West Bengal,10/12/21,Silver,True,05/06/24,STORE-055,PRD-6037,Fashion,Sneakers,1,2725.05,2725.05,Upi,SALE,2024-05-06T10:41:00.000Z,WEBSITE
TXN-00000009,CUST-00044,Kavya Verma,kavya.verma22@rediffmail.com,95318-84396,F,07-12-1978,Jaipur,Rajasthan,02-07-2023,Bronze,Y,14-02-2024,ONLINE,PRD-6531,NULL,Power Bank,4,1472.47,5889.88,wallet,EXCHANGE,2024-02-14T20:56:00.000Z,MOBILE_APP
TXN-00000010,CUST-01634,Kavya Gupta,kavya.gupta609@gmail.com,+91-9977283871,female,15/04/2005,Kolkata,West Bengal,2021-09-12,null,1,06/26/24,ONLINE,PRD-4677,Electronics,Smart Watch,5,3713.75,18568.75,COD,SALE,2024-06-26T17:32:00.000Z,POS


In [0]:
expected_columns = [
    "transaction_id", "customer_id", "full_name", "email", "phone", "gender",
    "date_of_birth", "city", "state", "registration_date", "loyalty_tier",
    "is_active", "transaction_date", "store_id", "product_id", "product_category",
    "item", "quantity", "unit_price", "total_amount", "payment_method",
    "transaction_type", "created_at", "source_system"
]

actual_columns = raw_df.columns

missing_columns = sorted(list(set(expected_columns) - set(actual_columns)))
extra_columns = sorted(list(set(actual_columns) - set(expected_columns)))

print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)

if missing_columns:
    raise Exception(f"Schema validation failed. Missing columns: {missing_columns}")
else:
    print("Schema validation passed successfully.")

Missing columns: []
Extra columns: []
Schema validation passed successfully.


In [0]:
raw_with_metadata_df = (
    raw_df
    .withColumn("run_id", F.lit(run_id))
    .withColumn("load_timestamp", F.current_timestamp())
    .withColumn("source_file_name", F.lit("trustguard_retail_transactions.csv"))
)

raw_with_metadata_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.raw_transactions")

print("Raw Layer table created successfully.")
print("Raw records:", raw_with_metadata_df.count())

display(spark.table("trustguard_db.raw_transactions").limit(10))

Raw Layer table created successfully.
Raw records: 10250


transaction_id,customer_id,full_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_tier,is_active,transaction_date,store_id,product_id,product_category,item,quantity,unit_price,total_amount,payment_method,transaction_type,created_at,source_system,run_id,load_timestamp,source_file_name
TXN-00000001,CUST-00951,Karan Sinha,karan.sinha340@gmail.com,+91-6478805564,M,05-17-1993,Bangalore,Karnataka,10/20/22,Bronze,Yes,2024-06-26,ONLINE,PRD-2441,Home & Kitchen,Bedsheet,2,1149.09,2298.18,UPI,SALE,2024-06-26T20:17:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000002,CUST-00072,Sahil Patel,sahil.patel859@rediffmail.com,7178884276,MALE,11-25-1993,Indore,Madhya Pradesh,2023-09-24,Platinum,TRUE,04/28/24,STORE-044,PRD-3426,Sports,Yoga Mat,5,721.0,3605.0,Net Banking,SALE,2024-04-28T09:02:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000003,CUST-01036,Harsh Sharma,harsh.sharma622@outlook.com,79596-10945,M,03-12-1992,Hyderabad,Telangana,13-07-2023,Platinum,Y,06/24/24,STORE-010,PRD-6531,Electronics,Power Bank,4,1503.12,6012.48,wallet,REFUND,2024-06-24T22:01:00.000Z,CRM_EXPORT,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000004,CUST-01382,Sakshi Sharma,sakshi.sharma56@yahoo.com,76711-28657,FEMALE,02-25-2000,Jaipur,Rajasthan,2023-04-07,Silver,1,2024-04-23,ONLINE,PRD-6028,Home & Kitchen,Pressure Cooker,4,1992.0,7968.0,wallet,SALE,2024-04-23T19:46:00.000Z,WEBSITE,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000005,CUST-02194,Priya Joshi,priya.joshi793@yahoo.com,99930-16643,female,06/04/1985,Bangalore,Karnataka,01-09-2021,Silver,Y,02/12/24,ONLINE,PRD-7016,Stationery,Gel Pen Set,2,139.41,278.82,COD,SALE,2024-02-12T14:39:00.000Z,CRM_EXPORT,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000006,CUST-00563,Isha Iyer,NULL,6779355497,FEMALE,10/01/1997,Bangalore,Karnataka,12-04-2021,Platinum,1,09-06-2024,ONLINE,PRD-9849,Groceries,Wheat Flour 10kg,5,486.53,2432.65,Credit Card,SALE,2024-06-09T13:09:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000007,CUST-02027,Karan Joshi,karan.joshi135@rediffmail.com,+91-6109084830,Male,1994-07-04,Bhopal,Madhya Pradesh,21-03-2021,Silver,Yes,2024-01-14,ONLINE,PRD-9228,Beauty,Body Lotion,1,255.62,255.62,NetBanking,SALE,2024-01-14T15:02:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000008,CUST-00559,Varun Das,varun.das931@rediffmail.com,7455984805,M,11-15-2005,Kolkata,West Bengal,10/12/21,Silver,True,05/06/24,STORE-055,PRD-6037,Fashion,Sneakers,1,2725.05,2725.05,Upi,SALE,2024-05-06T10:41:00.000Z,WEBSITE,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000009,CUST-00044,Kavya Verma,kavya.verma22@rediffmail.com,95318-84396,F,07-12-1978,Jaipur,Rajasthan,02-07-2023,Bronze,Y,14-02-2024,ONLINE,PRD-6531,NULL,Power Bank,4,1472.47,5889.88,wallet,EXCHANGE,2024-02-14T20:56:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000010,CUST-01634,Kavya Gupta,kavya.gupta609@gmail.com,+91-9977283871,female,15/04/2005,Kolkata,West Bengal,2021-09-12,null,1,06/26/24,ONLINE,PRD-4677,Electronics,Smart Watch,5,3713.75,18568.75,COD,SALE,2024-06-26T17:32:00.000Z,POS,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv


In [0]:
raw_transactions = spark.table("trustguard_db.raw_transactions")
total_records = raw_transactions.count()

dq_rows = []

important_columns = [
    "transaction_id",
    "customer_id",
    "product_category",
    "payment_method",
    "quantity",
    "unit_price",
    "total_amount",
    "transaction_date"
]

for column_name in important_columns:
    failed_count = raw_transactions.filter(
        F.col(column_name).isNull() | (F.trim(F.col(column_name).cast("string")) == "")
    ).count()

    passed_count = total_records - failed_count

    dq_rows.append((
        run_id,
        "raw_transactions",
        f"null_check_{column_name}",
        int(total_records),
        int(passed_count),
        int(failed_count),
        round((passed_count / total_records) * 100, 2),
        run_timestamp
    ))

duplicate_records = (
    raw_transactions
    .filter(F.col("transaction_id").isNotNull() & (F.trim(F.col("transaction_id").cast("string")) != ""))
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .agg(F.sum(F.col("count") - 1).alias("duplicate_records"))
    .collect()[0]["duplicate_records"]
)

duplicate_records = int(duplicate_records or 0)

dq_rows.append((
    run_id,
    "raw_transactions",
    "duplicate_transaction_id_check",
    int(total_records),
    int(total_records - duplicate_records),
    int(duplicate_records),
    round(((total_records - duplicate_records) / total_records) * 100, 2),
    run_timestamp
))

dq_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("check_name", StringType(), True),
    StructField("total_records", IntegerType(), True),
    StructField("passed_records", IntegerType(), True),
    StructField("failed_records", IntegerType(), True),
    StructField("dq_score", DoubleType(), True),
    StructField("run_timestamp", StringType(), True)
])

dq_report_df = spark.createDataFrame(dq_rows, dq_schema)

dq_report_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.dq_report")

display(dq_report_df)

run_id,table_name,check_name,total_records,passed_records,failed_records,dq_score,run_timestamp
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_transaction_id,10250,10225,25,99.76,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_customer_id,10250,10228,22,99.79,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_product_category,10250,9810,440,95.71,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_payment_method,10250,10155,95,99.07,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_quantity,10250,10250,0,100.0,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_unit_price,10250,10250,0,100.0,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_total_amount,10250,10250,0,100.0,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_transaction_date,10250,10250,0,100.0,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,duplicate_transaction_id_check,10250,10000,250,97.56,2026-07-11 15:22:55


In [0]:
clean_base_df = (
    raw_transactions
    .withColumn("transaction_id", F.trim(F.col("transaction_id").cast("string")))
    .withColumn("customer_id", F.trim(F.col("customer_id").cast("string")))
    .withColumn("full_name", F.initcap(F.trim(F.col("full_name").cast("string"))))
    .withColumn("email", F.lower(F.trim(F.col("email").cast("string"))))
    .withColumn("phone", F.regexp_replace(F.col("phone").cast("string"), "[^0-9]", ""))
    .withColumn("gender_raw", F.lower(F.trim(F.col("gender").cast("string"))))
    .withColumn("city_raw", F.lower(F.trim(F.col("city").cast("string"))))
    .withColumn("payment_method_raw", F.lower(F.trim(F.col("payment_method").cast("string"))))
    .withColumn(
        "product_category",
        F.when(
            F.col("product_category").isNull() | (F.trim(F.col("product_category").cast("string")) == ""),
            "Unknown"
        ).otherwise(F.initcap(F.trim(F.col("product_category").cast("string"))))
    )
    .withColumn("quantity_int", F.col("quantity").cast("int"))
    .withColumn("unit_price_double", F.col("unit_price").cast("double"))
    .withColumn("total_amount_double", F.col("total_amount").cast("double"))
)

clean_standardized_df = (
    clean_base_df
    .withColumn(
        "gender",
        F.when(F.col("gender_raw").isin("f", "female"), "Female")
         .when(F.col("gender_raw").isin("m", "male"), "Male")
         .otherwise("Unknown")
    )
    .withColumn(
        "city",
        F.when(F.col("city_raw").isin("mumabi", "mumbai"), "Mumbai")
         .when(F.col("city_raw").isin("dhelhi", "delhi"), "Delhi")
         .when(F.col("city_raw").isin("bangalor", "bangalore", "bengaluru"), "Bangalore")
         .when(F.col("city_raw").isin("chennia", "chennai"), "Chennai")
         .when(F.col("city_raw").isin("hydrabad", "hyderabad"), "Hyderabad")
         .otherwise(F.initcap(F.col("city_raw")))
    )
    .withColumn(
        "payment_method",
        F.when(F.col("payment_method_raw").isin("upi", "u.p.i.", "u p i"), "UPI")
         .when(F.col("payment_method_raw").isin("cash"), "CASH")
         .when(F.col("payment_method_raw").isin("card", "credit card", "debit card"), "CARD")
         .when(F.col("payment_method_raw").isin("cod", "cash on delivery"), "COD")
         .when(F.col("payment_method_raw").isin("net banking", "netbanking"), "NET_BANKING")
         .otherwise("UNKNOWN")
    )
    .withColumn(
        "is_active",
        F.when(F.lower(F.trim(F.col("is_active").cast("string"))).isin("true", "1", "yes", "active"), "Active")
         .when(F.lower(F.trim(F.col("is_active").cast("string"))).isin("false", "0", "no", "inactive"), "Inactive")
         .otherwise("Unknown")
    )
)

display(clean_standardized_df.limit(10))

transaction_id,customer_id,full_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_tier,is_active,transaction_date,store_id,product_id,product_category,item,quantity,unit_price,total_amount,payment_method,transaction_type,created_at,source_system,run_id,load_timestamp,source_file_name,gender_raw,city_raw,payment_method_raw,quantity_int,unit_price_double,total_amount_double
TXN-00000001,CUST-00951,Karan Sinha,karan.sinha340@gmail.com,916478805564,Male,05-17-1993,Bangalore,Karnataka,10/20/22,Bronze,Active,2024-06-26,ONLINE,PRD-2441,Home & Kitchen,Bedsheet,2,1149.09,2298.18,UPI,SALE,2024-06-26T20:17:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv,m,bangalore,upi,2,1149.09,2298.18
TXN-00000002,CUST-00072,Sahil Patel,sahil.patel859@rediffmail.com,7178884276,Male,11-25-1993,Indore,Madhya Pradesh,2023-09-24,Platinum,Active,04/28/24,STORE-044,PRD-3426,Sports,Yoga Mat,5,721.0,3605.0,NET_BANKING,SALE,2024-04-28T09:02:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv,male,indore,net banking,5,721.0,3605.0
TXN-00000003,CUST-01036,Harsh Sharma,harsh.sharma622@outlook.com,7959610945,Male,03-12-1992,Hyderabad,Telangana,13-07-2023,Platinum,Unknown,06/24/24,STORE-010,PRD-6531,Electronics,Power Bank,4,1503.12,6012.48,UNKNOWN,REFUND,2024-06-24T22:01:00.000Z,CRM_EXPORT,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv,m,hyderabad,wallet,4,1503.12,6012.48
TXN-00000004,CUST-01382,Sakshi Sharma,sakshi.sharma56@yahoo.com,7671128657,Female,02-25-2000,Jaipur,Rajasthan,2023-04-07,Silver,Active,2024-04-23,ONLINE,PRD-6028,Home & Kitchen,Pressure Cooker,4,1992.0,7968.0,UNKNOWN,SALE,2024-04-23T19:46:00.000Z,WEBSITE,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv,female,jaipur,wallet,4,1992.0,7968.0
TXN-00000005,CUST-02194,Priya Joshi,priya.joshi793@yahoo.com,9993016643,Female,06/04/1985,Bangalore,Karnataka,01-09-2021,Silver,Unknown,02/12/24,ONLINE,PRD-7016,Stationery,Gel Pen Set,2,139.41,278.82,COD,SALE,2024-02-12T14:39:00.000Z,CRM_EXPORT,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv,female,bangalore,cod,2,139.41,278.82
TXN-00000006,CUST-00563,Isha Iyer,null,6779355497,Female,10/01/1997,Bangalore,Karnataka,12-04-2021,Platinum,Active,09-06-2024,ONLINE,PRD-9849,Groceries,Wheat Flour 10kg,5,486.53,2432.65,CARD,SALE,2024-06-09T13:09:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv,female,bangalore,credit card,5,486.53,2432.65
TXN-00000007,CUST-02027,Karan Joshi,karan.joshi135@rediffmail.com,916109084830,Male,1994-07-04,Bhopal,Madhya Pradesh,21-03-2021,Silver,Active,2024-01-14,ONLINE,PRD-9228,Beauty,Body Lotion,1,255.62,255.62,NET_BANKING,SALE,2024-01-14T15:02:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv,male,bhopal,netbanking,1,255.62,255.62
TXN-00000008,CUST-00559,Varun Das,varun.das931@rediffmail.com,7455984805,Male,11-15-2005,Kolkata,West Bengal,10/12/21,Silver,Active,05/06/24,STORE-055,PRD-6037,Fashion,Sneakers,1,2725.05,2725.05,UPI,SALE,2024-05-06T10:41:00.000Z,WEBSITE,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv,m,kolkata,upi,1,2725.05,2725.05
TXN-00000009,CUST-00044,Kavya Verma,kavya.verma22@rediffmail.com,9531884396,Female,07-12-1978,Jaipur,Rajasthan,02-07-2023,Bronze,Unknown,14-02-2024,ONLINE,PRD-6531,Null,Power Bank,4,1472.47,5889.88,UNKNOWN,EXCHANGE,2024-02-14T20:56:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv,f,jaipur,wallet,4,1472.47,5889.88
TXN-00000010,CUST-01634,Kavya Gupta,kavya.gupta609@gmail.com,919977283871,Female,15/04/2005,Kolkata,West Bengal,2021-09-12,null,Active,06/26/24,ONLINE,PRD

In [0]:
# Cell 9: Standardize date columns safely

clean_dates_df = (
    clean_standardized_df
    .withColumn(
        "transaction_date_clean",
        F.coalesce(
            F.expr("try_to_date(CAST(transaction_date AS STRING), 'yyyy-MM-dd')"),
            F.expr("try_to_date(CAST(transaction_date AS STRING), 'dd-MM-yyyy')"),
            F.expr("try_to_date(CAST(transaction_date AS STRING), 'MM/dd/yy')"),
            F.expr("try_to_date(CAST(transaction_date AS STRING), 'MM/dd/yyyy')"),
            F.expr("try_to_date(CAST(transaction_date AS STRING), 'dd/MM/yyyy')")
        )
    )
    .withColumn(
        "date_of_birth_clean",
        F.coalesce(
            F.expr("try_to_date(CAST(date_of_birth AS STRING), 'yyyy-MM-dd')"),
            F.expr("try_to_date(CAST(date_of_birth AS STRING), 'dd-MM-yyyy')"),
            F.expr("try_to_date(CAST(date_of_birth AS STRING), 'MM-dd-yyyy')"),
            F.expr("try_to_date(CAST(date_of_birth AS STRING), 'dd/MM/yyyy')"),
            F.expr("try_to_date(CAST(date_of_birth AS STRING), 'MM/dd/yyyy')")
        )
    )
    .withColumn(
        "registration_date_clean",
        F.coalesce(
            F.expr("try_to_date(CAST(registration_date AS STRING), 'yyyy-MM-dd')"),
            F.expr("try_to_date(CAST(registration_date AS STRING), 'dd-MM-yyyy')"),
            F.expr("try_to_date(CAST(registration_date AS STRING), 'MM-dd-yyyy')"),
            F.expr("try_to_date(CAST(registration_date AS STRING), 'dd/MM/yyyy')"),
            F.expr("try_to_date(CAST(registration_date AS STRING), 'MM/dd/yyyy')")
        )
    )
    .withColumn(
        "created_at_clean",
        F.expr("try_to_timestamp(CAST(created_at AS STRING))")
    )
)

display(
    clean_dates_df
    .select(
        "transaction_date",
        "transaction_date_clean",
        "date_of_birth",
        "date_of_birth_clean",
        "registration_date",
        "registration_date_clean"
    )
    .limit(20)
)

transaction_date,transaction_date_clean,date_of_birth,date_of_birth_clean,registration_date,registration_date_clean
2024-06-26,2024-06-26,05-17-1993,1993-05-17,10/20/22,null
04/28/24,2024-04-28,11-25-1993,1993-11-25,2023-09-24,2023-09-24
06/24/24,2024-06-24,03-12-1992,1992-12-03,13-07-2023,2023-07-13
2024-04-23,2024-04-23,02-25-2000,2000-02-25,2023-04-07,2023-04-07
02/12/24,2024-02-12,06/04/1985,1985-04-06,01-09-2021,2021-09-01
09-06-2024,2024-06-09,10/01/1997,1997-01-10,12-04-2021,2021-04-12
2024-01-14,2024-01-14,1994-07-04,1994-07-04,21-03-2021,2021-03-21
05/06/24,2024-05-06,11-15-2005,2005-11-15,10/12/21,null
14-02-2024,2024-02-14,07-12-1978,1978-12-07,02-07-2023,2023-07-02
06/26/24,2024-06-26,15/04/2005,2005-04-15,2021-09-12,2021-09-12


In [0]:
validated_df = (
    clean_dates_df
    .withColumn("expected_total_amount", F.round(F.col("quantity_int") * F.col("unit_price_double"), 2))
    .withColumn("amount_difference", F.abs(F.col("total_amount_double") - F.col("expected_total_amount")))
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(F.col("transaction_id").isNull() | (F.col("transaction_id") == ""), "Missing transaction_id"),
            F.when(F.col("customer_id").isNull() | (F.col("customer_id") == ""), "Missing customer_id"),
            F.when(F.col("quantity_int").isNull(), "Invalid quantity"),
            F.when(F.col("quantity_int") <= 0, "Non-positive quantity"),
            F.when(F.col("unit_price_double").isNull(), "Invalid unit price"),
            F.when(F.col("unit_price_double") <= 0, "Non-positive unit price"),
            F.when(F.col("transaction_date_clean").isNull(), "Invalid transaction date"),
            F.when(F.col("amount_difference") > 1, "Total amount mismatch")
        )
    )
)

rejected_records_df = (
    validated_df
    .filter(F.col("rejection_reason") != "")
    .withColumn("rejected_at", F.current_timestamp())
    .withColumn("run_id", F.lit(run_id))
)

rejected_records_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.rejected_records")

print("Rejected records:", rejected_records_df.count())
display(rejected_records_df.select("transaction_id", "customer_id", "rejection_reason").limit(20))

Rejected records: 338


transaction_id,customer_id,rejection_reason
TXN-00000033,CUST-00668,Total amount mismatch
TXN-00000065,CUST-02170,Total amount mismatch
TXN-00000076,null,Missing customer_id
TXN-00000126,CUST-01945,Non-positive quantity
TXN-00000146,CUST-01699,Non-positive quantity
TXN-00000162,CUST-01460,Non-positive quantity
TXN-00000204,CUST-00178,Non-positive quantity
TXN-00000281,CUST-01919,Non-positive quantity
TXN-00000386,CUST-01577,Total amount mismatch
TXN-00000391,CUST-01023,Total amount mismatch


In [0]:
valid_clean_df = validated_df.filter(F.col("rejection_reason") == "")

window_spec = Window.partitionBy("transaction_id").orderBy(F.col("created_at_clean").asc_nulls_last())

deduped_df = (
    valid_clean_df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

clean_transactions_df = deduped_df.select(
    "transaction_id",
    "customer_id",
    "full_name",
    "email",
    "phone",
    "gender",
    F.col("date_of_birth_clean").alias("date_of_birth"),
    "city",
    "state",
    F.col("registration_date_clean").alias("registration_date"),
    "loyalty_tier",
    "is_active",
    F.col("transaction_date_clean").alias("transaction_date"),
    "store_id",
    "product_id",
    "product_category",
    "item",
    F.col("quantity_int").alias("quantity"),
    F.col("unit_price_double").alias("unit_price"),
    F.col("expected_total_amount").alias("total_amount"),
    "payment_method",
    "transaction_type",
    F.col("created_at_clean").alias("created_at"),
    "source_system",
    "run_id",
    "load_timestamp",
    "source_file_name"
)

clean_transactions_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.clean_transactions")

print("Clean transactions:", clean_transactions_df.count())
display(clean_transactions_df.limit(10))

Clean transactions: 9668


transaction_id,customer_id,full_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_tier,is_active,transaction_date,store_id,product_id,product_category,item,quantity,unit_price,total_amount,payment_method,transaction_type,created_at,source_system,run_id,load_timestamp,source_file_name
TXN-00000001,CUST-00951,Karan Sinha,karan.sinha340@gmail.com,916478805564,Male,1993-05-17,Bangalore,Karnataka,null,Bronze,Active,2024-06-26,ONLINE,PRD-2441,Home & Kitchen,Bedsheet,2,1149.09,2298.18,UPI,SALE,2024-06-26T20:17:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000002,CUST-00072,Sahil Patel,sahil.patel859@rediffmail.com,7178884276,Male,1993-11-25,Indore,Madhya Pradesh,2023-09-24,Platinum,Active,2024-04-28,STORE-044,PRD-3426,Sports,Yoga Mat,5,721.0,3605.0,NET_BANKING,SALE,2024-04-28T09:02:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000003,CUST-01036,Harsh Sharma,harsh.sharma622@outlook.com,7959610945,Male,1992-12-03,Hyderabad,Telangana,2023-07-13,Platinum,Unknown,2024-06-24,STORE-010,PRD-6531,Electronics,Power Bank,4,1503.12,6012.48,UNKNOWN,REFUND,2024-06-24T22:01:00.000Z,CRM_EXPORT,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000004,CUST-01382,Sakshi Sharma,sakshi.sharma56@yahoo.com,7671128657,Female,2000-02-25,Jaipur,Rajasthan,2023-04-07,Silver,Active,2024-04-23,ONLINE,PRD-6028,Home & Kitchen,Pressure Cooker,4,1992.0,7968.0,UNKNOWN,SALE,2024-04-23T19:46:00.000Z,WEBSITE,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000005,CUST-02194,Priya Joshi,priya.joshi793@yahoo.com,9993016643,Female,1985-04-06,Bangalore,Karnataka,2021-09-01,Silver,Unknown,2024-02-12,ONLINE,PRD-7016,Stationery,Gel Pen Set,2,139.41,278.82,COD,SALE,2024-02-12T14:39:00.000Z,CRM_EXPORT,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000006,CUST-00563,Isha Iyer,null,6779355497,Female,1997-01-10,Bangalore,Karnataka,2021-04-12,Platinum,Active,2024-06-09,ONLINE,PRD-9849,Groceries,Wheat Flour 10kg,5,486.53,2432.65,CARD,SALE,2024-06-09T13:09:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000007,CUST-02027,Karan Joshi,karan.joshi135@rediffmail.com,916109084830,Male,1994-07-04,Bhopal,Madhya Pradesh,2021-03-21,Silver,Active,2024-01-14,ONLINE,PRD-9228,Beauty,Body Lotion,1,255.62,255.62,NET_BANKING,SALE,2024-01-14T15:02:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000008,CUST-00559,Varun Das,varun.das931@rediffmail.com,7455984805,Male,2005-11-15,Kolkata,West Bengal,null,Silver,Active,2024-05-06,STORE-055,PRD-6037,Fashion,Sneakers,1,2725.05,2725.05,UPI,SALE,2024-05-06T10:41:00.000Z,WEBSITE,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000009,CUST-00044,Kavya Verma,kavya.verma22@rediffmail.com,9531884396,Female,1978-12-07,Jaipur,Rajasthan,2023-07-02,Bronze,Unknown,2024-02-14,ONLINE,PRD-6531,Null,Power Bank,4,1472.47,5889.88,UNKNOWN,EXCHANGE,2024-02-14T20:56:00.000Z,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv
TXN-00000010,CUST-01634,Kavya Gupta,kavya.gupta609@gmail.com,919977283871,Female,2005-04-15,Kolkata,West Bengal,2021-09-12,null,Active,2024-06-26,ONLINE,PRD-4677,Electronics,Smart Watch,5,3713.75,18568.75,COD,SALE,2024-06-26T17:32:00.000Z,POS,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11T15:44:55.418Z,trustguard_retail_transactions.csv


In [0]:
customer_window = Window.partitionBy("customer_id").orderBy(F.col("registration_date").asc_nulls_last())

clean_customers_df = (
    clean_transactions_df
    .select(
        "customer_id",
        "full_name",
        "email",
        "phone",
        "gender",
        "date_of_birth",
        "city",
        "state",
        "registration_date",
        "loyalty_tier",
        "is_active"
    )
    .filter(F.col("customer_id").isNotNull())
    .withColumn("row_num", F.row_number().over(customer_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

clean_customers_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.clean_customers")

print("Clean customers:", clean_customers_df.count())
display(clean_customers_df.limit(10))

Clean customers: 2232


customer_id,full_name,email,phone,gender,date_of_birth,city,state,registration_date,loyalty_tier,is_active
CUST-00001,Aarav Joshi,aarav.joshi152@gmail.com,8906402157,Male,1993-07-02,Hyderabad,Telangana,2021-04-06,null,Active
CUST-00002,Nisha Joshi,nisha.joshi469@outlook.com,9476477323,Female,1999-03-23,Jaipur,Rajasthan,2021-12-15,Platinum,Active
CUST-00003,Isha Verma,isha.verma628@outlook.com,9466589567,Female,1998-08-18,Pune,Maharashtra,2023-09-24,null,Active
CUST-00004,Karan Das,karan.das56@yahoo.com,919320303242,Male,1977-04-28,Chennai,Tamil Nadu,2022-01-25,Gold,Active
CUST-00005,Neha Iyer,neha.iyer709@gmail.com,918616197747,Female,1992-12-08,Hyderabad,Telangana,2022-04-19,Platinum,Active
CUST-00006,Aarav Gupta,null,917722989659,Male,1977-04-19,Patna,Bihar,2021-11-19,Bronze,Unknown
CUST-00007,Varun Mehta,varun.mehta262@outlook.com,9208399878,Male,2003-10-13,Bangalore,Karnataka,2021-08-13,Silver,Active
CUST-00008,Rahul Singh,rahul.singh706@rediffmail.com,8561557300,Male,1987-07-20,Surat,Gujarat,2022-06-26,Silver,Active
CUST-00009,Nikhil Sinha,nikhil.sinha358@gmail.com,7260573448,Male,1980-01-08,Hyderabad,Telangana,null,null,Active
CUST-00010,Varun Patel,varun.patel633@yahoo.com,916656439677,Male,1999-03-18,Patna,Bihar,2023-07-30,Bronze,Active


In [0]:
final_transactions_df = (
    clean_transactions_df.alias("t")
    .join(clean_customers_df.alias("c"), on="customer_id", how="inner")
    .select(
        F.col("t.transaction_id"),
        F.col("t.customer_id"),
        F.col("c.full_name"),
        F.col("c.email"),
        F.col("c.phone"),
        F.col("c.gender"),
        F.col("c.city"),
        F.col("c.state"),
        F.col("c.loyalty_tier"),
        F.col("c.is_active"),
        F.col("t.transaction_date"),
        F.col("t.store_id"),
        F.col("t.product_id"),
        F.col("t.product_category"),
        F.col("t.item"),
        F.col("t.quantity"),
        F.col("t.unit_price"),
        F.col("t.total_amount"),
        F.col("t.payment_method"),
        F.col("t.transaction_type"),
        F.col("t.source_system"),
        F.col("t.run_id")
    )
)

final_transactions_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.final_transactions")

print("Final transactions:", final_transactions_df.count())
display(final_transactions_df.limit(10))

Final transactions: 9668


transaction_id,customer_id,full_name,email,phone,gender,city,state,loyalty_tier,is_active,transaction_date,store_id,product_id,product_category,item,quantity,unit_price,total_amount,payment_method,transaction_type,source_system,run_id
TXN-00000001,CUST-00951,Karan Sinha,karan.sinha340@gmail.com,916478805564,Male,Bangalore,Karnataka,Bronze,Active,2024-06-26,ONLINE,PRD-2441,Home & Kitchen,Bedsheet,2,1149.09,2298.18,UPI,SALE,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19
TXN-00000002,CUST-00072,Sahil Patel,sahil.patel859@rediffmail.com,7178884276,Male,Indore,Madhya Pradesh,Platinum,Active,2024-04-28,STORE-044,PRD-3426,Sports,Yoga Mat,5,721.0,3605.0,NET_BANKING,SALE,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19
TXN-00000003,CUST-01036,Harsh Sharma,harsh.sharma622@outlook.com,7959610945,Male,Hyderabad,Telangana,Platinum,Unknown,2024-06-24,STORE-010,PRD-6531,Electronics,Power Bank,4,1503.12,6012.48,UNKNOWN,REFUND,CRM_EXPORT,9c3a1eb1-248a-4654-a0d6-b7331ee07f19
TXN-00000004,CUST-01382,Sakshi Sharma,sakshi.sharma56@yahoo.com,7671128657,Female,Jaipur,Rajasthan,Silver,Active,2024-04-23,ONLINE,PRD-6028,Home & Kitchen,Pressure Cooker,4,1992.0,7968.0,UNKNOWN,SALE,WEBSITE,9c3a1eb1-248a-4654-a0d6-b7331ee07f19
TXN-00000005,CUST-02194,Priya Joshi,priya.joshi793@yahoo.com,9993016643,Female,Bangalore,Karnataka,Silver,Unknown,2024-02-12,ONLINE,PRD-7016,Stationery,Gel Pen Set,2,139.41,278.82,COD,SALE,CRM_EXPORT,9c3a1eb1-248a-4654-a0d6-b7331ee07f19
TXN-00000006,CUST-00563,Isha Iyer,null,6779355497,Female,Bangalore,Karnataka,Platinum,Active,2024-06-09,ONLINE,PRD-9849,Groceries,Wheat Flour 10kg,5,486.53,2432.65,CARD,SALE,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19
TXN-00000007,CUST-02027,Karan Joshi,karan.joshi135@rediffmail.com,916109084830,Male,Bhopal,Madhya Pradesh,Silver,Active,2024-01-14,ONLINE,PRD-9228,Beauty,Body Lotion,1,255.62,255.62,NET_BANKING,SALE,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19
TXN-00000008,CUST-00559,Varun Das,varun.das931@rediffmail.com,7455984805,Male,Kolkata,West Bengal,Silver,Active,2024-05-06,STORE-055,PRD-6037,Fashion,Sneakers,1,2725.05,2725.05,UPI,SALE,WEBSITE,9c3a1eb1-248a-4654-a0d6-b7331ee07f19
TXN-00000009,CUST-00044,Kavya Verma,kavya.verma22@rediffmail.com,9531884396,Female,Jaipur,Rajasthan,Bronze,Unknown,2024-02-14,ONLINE,PRD-6531,Null,Power Bank,4,1472.47,5889.88,UNKNOWN,EXCHANGE,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19
TXN-00000010,CUST-01634,Kavya Gupta,kavya.gupta609@gmail.com,919977283871,Female,Kolkata,West Bengal,null,Active,2024-06-26,ONLINE,PRD-4677,Electronics,Smart Watch,5,3713.75,18568.75,COD,SALE,POS,9c3a1eb1-248a-4654-a0d6-b7331ee07f19


In [0]:
customer_summary_df = (
    final_transactions_df
    .groupBy("customer_id", "full_name", "city", "state", "loyalty_tier")
    .agg(
        F.countDistinct("transaction_id").alias("total_orders"),
        F.round(F.sum("total_amount"), 2).alias("total_spent"),
        F.round(F.avg("total_amount"), 2).alias("average_order_value"),
        F.min("transaction_date").alias("first_purchase_date"),
        F.max("transaction_date").alias("last_purchase_date")
    )
)

customer_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.customer_summary")

display(customer_summary_df.limit(10))

customer_id,full_name,city,state,loyalty_tier,total_orders,total_spent,average_order_value,first_purchase_date,last_purchase_date
CUST-00001,Aarav Joshi,Hyderabad,Telangana,null,5,6521.36,1304.27,2024-01-09,2024-03-14
CUST-00002,Nisha Joshi,Jaipur,Rajasthan,Platinum,4,12619.72,3154.93,2024-03-12,2024-06-10
CUST-00003,Isha Verma,Pune,Maharashtra,null,4,16116.58,4029.15,2024-01-03,2024-06-23
CUST-00004,Karan Das,Chennai,Tamil Nadu,Gold,4,13732.27,3433.07,2024-03-03,2024-06-24
CUST-00005,Neha Iyer,Hyderabad,Telangana,Platinum,2,13896.28,6948.14,2024-06-02,2024-06-10
CUST-00006,Aarav Gupta,Patna,Bihar,Bronze,2,2329.74,1164.87,2024-01-24,2024-03-08
CUST-00007,Varun Mehta,Bangalore,Karnataka,Silver,4,8304.93,2076.23,2024-03-03,2024-05-05
CUST-00008,Rahul Singh,Surat,Gujarat,Silver,5,11578.2,2315.64,2024-02-20,2024-06-18
CUST-00009,Nikhil Sinha,Hyderabad,Telangana,null,5,11797.6,2359.52,2024-02-11,2024-06-11
CUST-00010,Varun Patel,Patna,Bihar,Bronze,6,7697.85,1282.98,2024-01-23,2024-04-29


In [0]:
city_sales_report_df = (
    final_transactions_df
    .withColumn("sales_month", F.date_format("transaction_date", "yyyy-MM"))
    .groupBy("city", "state", "sales_month")
    .agg(
        F.countDistinct("transaction_id").alias("total_orders"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("average_order_value")
    )
    .orderBy(F.col("total_revenue").desc())
)

city_sales_report_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.city_sales_report")

display(city_sales_report_df.limit(20))

city,state,sales_month,total_orders,total_revenue,average_order_value
Indore,Madhya Pradesh,2024-03,164,550331.51,3355.68
Lucknow,Uttar Pradesh,2024-03,133,500057.3,3759.83
Delhi,Delhi,2024-03,126,460482.45,3654.62
Pune,Maharashtra,2024-01,108,437468.38,4050.63
Chandigarh,Chandigarh,2024-05,126,435832.34,3458.99
Lucknow,Uttar Pradesh,2024-05,121,426717.44,3526.59
Hyderabad,Telangana,2024-05,115,414437.23,3603.8
Chennai,Tamil Nadu,2024-05,128,406095.5,3172.62
Surat,Gujarat,2024-04,114,405231.78,3554.66
Hyderabad,Telangana,2024-03,111,395934.95,3566.98


In [0]:
payment_method_report_df = (
    final_transactions_df
    .groupBy("payment_method")
    .agg(
        F.countDistinct("transaction_id").alias("total_transactions"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy(F.col("total_transactions").desc())
)

payment_method_report_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.payment_method_report")

display(payment_method_report_df)

payment_method,total_transactions,total_revenue
UNKNOWN,2258,7125928.05
CASH,1590,4770052.44
CARD,1576,4942820.24
NET_BANKING,1539,4916597.01
UPI,1532,4690878.79
COD,1173,3605877.18


In [0]:
product_category_report_df = (
    final_transactions_df
    .groupBy("product_category")
    .agg(
        F.countDistinct("transaction_id").alias("total_orders"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("average_order_value")
    )
    .orderBy(F.col("total_revenue").desc())
)

product_category_report_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.product_category_report")

display(product_category_report_df)

product_category,total_orders,total_revenue,average_order_value
Home & Kitchen,1331,6972410.81,5238.48
Sports,1276,6087061.14,4770.42
Electronics,1202,5304914.5,4413.41
Fashion,1300,5255193.56,4042.46
Beauty,1281,1552217.77,1211.72
Stationery,1227,1394902.93,1136.84
Groceries,1266,1236357.63,976.59
Unknown,422,1188657.25,2816.72
Null,363,1060438.12,2921.32


In [0]:
stats = final_transactions_df.agg(
    F.avg("quantity").alias("avg_quantity"),
    F.stddev("quantity").alias("std_quantity"),
    F.avg("total_amount").alias("avg_amount"),
    F.stddev("total_amount").alias("std_amount")
).collect()[0]

avg_quantity = stats["avg_quantity"]
std_quantity = stats["std_quantity"]
avg_amount = stats["avg_amount"]
std_amount = stats["std_amount"]

anomaly_df = (
    final_transactions_df
    .withColumn(
        "anomaly_reason",
        F.concat_ws(
            "; ",
            F.when(F.col("quantity") > avg_quantity + (3 * std_quantity), "Unusually high quantity"),
            F.when(F.col("total_amount") > avg_amount + (3 * std_amount), "Unusually high transaction amount")
        )
    )
    .filter(F.col("anomaly_reason") != "")
    .withColumn("detected_at", F.current_timestamp())
    .withColumn("run_id", F.lit(run_id))
)

anomaly_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.anomaly_log")

print("Anomaly records:", anomaly_df.count())
display(anomaly_df.limit(20))

Anomaly records: 240


transaction_id,customer_id,full_name,email,phone,gender,city,state,loyalty_tier,is_active,transaction_date,store_id,product_id,product_category,item,quantity,unit_price,total_amount,payment_method,transaction_type,source_system,run_id,anomaly_reason,detected_at
TXN-00000010,CUST-01634,Kavya Gupta,kavya.gupta609@gmail.com,919977283871,Female,Kolkata,West Bengal,null,Active,2024-06-26,ONLINE,PRD-4677,Electronics,Smart Watch,5,3713.75,18568.75,COD,SALE,POS,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,Unusually high transaction amount,2026-07-11T16:02:18.035Z
TXN-00000102,CUST-00308,Nisha Chopra,nisha.chopra395@yahoo.com,8217799930,Female,Delhi,Delhi,Bronze,Active,2024-05-03,STORE-039,PRD-4677,Electronics,Smart Watch,4,4108.47,16433.88,UNKNOWN,SALE,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,Unusually high transaction amount,2026-07-11T16:02:18.035Z
TXN-00000117,CUST-01053,Nikhil Reddy,nikhil.reddy78@yahoo.com,8500188574,Male,Jaipur,Rajasthan,Bronze,Active,2024-02-19,ONLINE,PRD-4677,Electronics,Smart Watch,5,3397.94,16989.7,NET_BANKING,SALE,POS,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,Unusually high transaction amount,2026-07-11T16:02:18.035Z
TXN-00000122,CUST-01721,Aarav Khan,aarav.khan532@rediffmail.com,6720043822,Male,Lucknow,Uttar Pradesh,Bronze,Active,2024-06-27,ONLINE,PRD-4677,Electronics,Smart Watch,5,3394.23,16971.15,NET_BANKING,SALE,POS,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,Unusually high transaction amount,2026-07-11T16:02:18.035Z
TXN-00000224,CUST-01218,Meera Khan,meera.khan366@rediffmail.com,916699175664,Female,Jaipur,Rajasthan,Bronze,Active,2024-03-03,ONLINE,PRD-6576,Sports,Sports Shoes,5,2877.32,14386.6,CARD,SALE,WEBSITE,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,Unusually high transaction amount,2026-07-11T16:02:18.035Z
TXN-00000238,CUST-01614,Rahul Khan,rahul.khan774@gmail.com,916256884011,Male,Pune,Maharashtra,Platinum,Active,2024-02-23,ONLINE,PRD-6037,Fashion,Sneakers,5,2911.28,14556.4,CASH,SALE,CRM_EXPORT,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,Unusually high transaction amount,2026-07-11T16:02:18.035Z
TXN-00000266,CUST-00937,Nisha Singh,nisha.singh643@yahoo.com,919621683021,Female,Surat,Gujarat,Bronze,Active,2024-02-17,ONLINE,PRD-3963,Home & Kitchen,Mixer Grinder,5,2981.33,14906.65,UPI,SALE,WEBSITE,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,Unusually high transaction amount,2026-07-11T16:02:18.035Z
TXN-00000325,CUST-01999,Priya Chopra,priya.chopra236@yahoo.com,8823975269,Female,Lucknow,Uttar Pradesh,null,Active,2024-02-27,STORE-020,PRD-4677,Electronics,Smart Watch,5,3084.95,15424.75,UNKNOWN,SALE,MOBILE_APP,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,Unusually high transaction amount,2026-07-11T16:02:18.035Z
TXN-00000327,CUST-02146,Arjun Reddy,arjun.reddy319@rediffmail.com,6896041477,Male,Bhopal,Madhya Pradesh,Platinum,Unknown,2024-05-22,STORE-032,PRD-3963,Home & Kitchen,Mixer Grinder,4,3802.78,15211.12,CARD,SALE,CRM_EXPORT,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,Unusually high transaction amount,2026-07-11T16:02:18.035Z
TXN-00000365,CUST-01090,Karan Chopra,karan.chopra274@outlook.com,918712377182,Male,Ahmedabad,Gujarat,null,Active,2024-04-15,ONLINE,PRD-6576,Sports,Sports Shoes,5,3121.25,15606.25,CARD,SALE,POS,9c3a1eb1-248a-4654-a0d6-b7331ee07f19,Unusually high transaction amount,2026-07-11T16:02:18.035Z


In [0]:
raw_count = raw_transactions.count()
clean_count = clean_transactions_df.count()
rejected_count = rejected_records_df.count()
final_count = final_transactions_df.count()
anomaly_count = anomaly_df.count()

pipeline_log_data = [
    (run_id, run_timestamp, "Raw", "raw_transactions", raw_count, raw_count, 0, "SUCCESS", None),
    (run_id, run_timestamp, "Clean", "clean_transactions", raw_count, clean_count, rejected_count, "SUCCESS", None),
    (run_id, run_timestamp, "Final", "final_transactions", clean_count, final_count, 0, "SUCCESS", None),
    (run_id, run_timestamp, "Report", "dq_report", raw_count, final_count, rejected_count, "SUCCESS", None),
    (run_id, run_timestamp, "Anomaly", "anomaly_log", final_count, anomaly_count, anomaly_count, "SUCCESS", None)
]

pipeline_log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("run_timestamp", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("records_in", IntegerType(), True),
    StructField("records_out", IntegerType(), True),
    StructField("records_rejected", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

pipeline_log_df = spark.createDataFrame(pipeline_log_data, pipeline_log_schema)

pipeline_log_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("trustguard_db.pipeline_log")

display(pipeline_log_df)

run_id,run_timestamp,layer,table_name,records_in,records_out,records_rejected,status,error_message
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11 15:22:55,Raw,raw_transactions,10250,10250,0,SUCCESS,null
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11 15:22:55,Clean,clean_transactions,10250,9668,338,SUCCESS,null
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11 15:22:55,Final,final_transactions,9668,9668,0,SUCCESS,null
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11 15:22:55,Report,dq_report,10250,9668,338,SUCCESS,null
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,2026-07-11 15:22:55,Anomaly,anomaly_log,9668,240,240,SUCCESS,null


In [0]:
%sql
SHOW TABLES IN trustguard_db;

database,tableName,isTemporary
trustguard_db,anomaly_log,false
trustguard_db,city_sales_report,false
trustguard_db,clean_customers,false
trustguard_db,clean_transactions,false
trustguard_db,customer_summary,false
trustguard_db,dq_report,false
trustguard_db,final_transactions,false
trustguard_db,payment_method_report,false
trustguard_db,pipeline_log,false
trustguard_db,product_category_report,false


In [0]:
%sql
SELECT 
    city,
    state,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    COUNT(DISTINCT transaction_id) AS total_orders
FROM trustguard_db.final_transactions
GROUP BY city, state
ORDER BY total_revenue DESC;

city,state,total_revenue,total_orders
Indore,Madhya Pradesh,2374908.99,738
Lucknow,Uttar Pradesh,2278559.31,663
Delhi,Delhi,2152365.25,638
Chandigarh,Chandigarh,2102485.34,682
Surat,Gujarat,2073856.62,695
Mumbai,Maharashtra,2071924.0,670
Chennai,Tamil Nadu,2065142.96,690
Hyderabad,Telangana,2046898.78,642
Kolkata,West Bengal,1942040.94,638
Pune,Maharashtra,1894530.82,569


In [0]:
%sql
SELECT 
    customer_id,
    full_name,
    city,
    ROUND(SUM(total_amount), 2) AS total_spent,
    COUNT(DISTINCT transaction_id) AS total_orders
FROM trustguard_db.final_transactions
GROUP BY customer_id, full_name, city
ORDER BY total_spent DESC
LIMIT 10;

customer_id,full_name,city,total_spent,total_orders
CUST-00732,Pooja Joshi,Jaipur,59398.23,7
CUST-00225,Nisha Nair,Bangalore,59299.06,10
CUST-01608,Riya Verma,Bhopal,57000.88,8
CUST-01052,Riya Iyer,Pune,56756.04,10
CUST-00454,Sahil Joshi,Lucknow,52962.65,6
CUST-00397,Kavya Reddy,Kolkata,52527.23,6
CUST-01918,Nisha Verma,Pune,51386.58,8
CUST-01053,Nikhil Reddy,Jaipur,49317.32,5
CUST-00216,Riya Joshi,Lucknow,49044.2,9
CUST-01565,Rohan Sharma,Indore,48223.93,9


In [0]:
%sql
SELECT 
    rejection_reason,
    COUNT(*) AS total_records
FROM trustguard_db.rejected_records
GROUP BY rejection_reason
ORDER BY total_records DESC;

rejection_reason,total_records
Non-positive quantity,165
Total amount mismatch,124
Missing transaction_id,25
Missing customer_id,22
Non-positive quantity; Total amount mismatch,2


In [0]:
%sql
SELECT *
FROM trustguard_db.dq_report
ORDER BY failed_records DESC;

run_id,table_name,check_name,total_records,passed_records,failed_records,dq_score,run_timestamp
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_product_category,10250,9810,440,95.71,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,duplicate_transaction_id_check,10250,10000,250,97.56,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_payment_method,10250,10155,95,99.07,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_transaction_id,10250,10225,25,99.76,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_customer_id,10250,10228,22,99.79,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_quantity,10250,10250,0,100.0,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_unit_price,10250,10250,0,100.0,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_total_amount,10250,10250,0,100.0,2026-07-11 15:22:55
9c3a1eb1-248a-4654-a0d6-b7331ee07f19,raw_transactions,null_check_transaction_date,10250,10250,0,100.0,2026-07-11 15:22:55
